# BC카드 공모전 — 1단계 전처리 (외국인 세그먼트)

원본 데이터: `ABP_CONTEST_DATA.csv` (242,574행, 2026.1~6월, 시군구 233개 × 업종 11개)

이 노트북은 파이프라인 1단계(전처리)를 수행한다:
1. 데이터 로드 및 검증
2. 업종명 공백 정규화 (편의점 → "편 의 점" 같은 표기 오류 수정)
3. 외국인(GENDER_CD=3) 세그먼트 추출
4. 시군구×업종×월 피벗 테이블 구성 (AMT, CNT)
5. 전국 레벨 baseline 집계 (상대성장률 계산용)
6. 최소 거래건수 필터 적용 (노이즈 셀 제거)

결과는 2단계(EDA)·3단계(신호 재랭킹)·4단계(가설검증)에서 그대로 이어받아 쓸 수 있도록 저장한다.


## 0. 환경 설정 & 파일 업로드

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# 방법 B: 구글 드라이브 마운트 (드라이브의 BC카드 폴더에 CSV를 미리 올려둔 경우) — 기본값
from google.colab import drive
drive.mount('/content/drive')
FILE_PATH = '/content/drive/MyDrive/BC카드/ABP_CONTEST_DATA.csv'

# 방법 A: 드라이브에 파일이 없거나 다른 파일로 테스트할 때는 이쪽을 대신 사용 (팝업에서 직접 업로드)
# from google.colab import files
# uploaded = files.upload()
# FILE_PATH = list(uploaded.keys())[0]

import os
assert os.path.exists(FILE_PATH), f"파일을 찾을 수 없음: {FILE_PATH} — 드라이브에 실제로 파일이 있는지, 폴더명/경로가 정확한지 확인"
print("사용할 파일:", FILE_PATH)


## 1. 데이터 로드

지난번에 확인한 것: 원본 CSV는 UTF-8로 정상 인코딩되어 있음(엑셀에서 깨져 보이는 건 BOM 없는 UTF-8을 엑셀이 CP949로 오인식하는 흔한 현상일 뿐, 실제 인코딩 문제 아님). 다만 컬럼 dtype은 명시적으로 지정한다 — GENDER_CD/AGE_CD/TP_BUZ_NO는 숫자처럼 보여도 'x'(법인) 같은 문자 값이 섞여 있어서 문자열로 읽어야 안전하다.

In [ ]:
dtype_map = {
    'STRD_YYMM': str,
    'SIDO_NM': str,
    'CCG_NM': str,
    'GENDER_CD': str,
    'AGE_CD': str,
    'TP_BUZ_NO': str,
    'TP_BUZ_NM': str,
}

df = pd.read_csv(FILE_PATH, encoding='utf-8', dtype=dtype_map)

# 원본 컬럼명이 소문자 amt/cnt로 되어있음(공모전 안내 문서의 표기는 AMT/CNT) → 문서 표기에 맞춰 통일
df = df.rename(columns={'amt': 'AMT', 'cnt': 'CNT'})
df['AMT'] = df['AMT'].astype('int64')
df['CNT'] = df['CNT'].astype('int64')

print(f"행 개수: {len(df):,}")
df.head()


## 2. 데이터 검증

In [ ]:
print("기간:", sorted(df['STRD_YYMM'].unique()))
print("시도 개수:", df['SIDO_NM'].nunique())
print("시군구 개수:", df['CCG_NM'].nunique())
print("업종 개수:", df['TP_BUZ_NM'].nunique())
print()
print("결측치:")
print(df.isnull().sum())
print()
print("성별 코드 분포:")
print(df['GENDER_CD'].value_counts())
print()
print("⚠️ 데이터 정의 오차 확인: 안내서상 'AGE_CD=X는 법인·외국인'이라 되어있으나,")
print("   실제로는 법인(GENDER_CD='x')만 AGE_CD='x'이고 외국인(GENDER_CD='3')은 실제 연령대(1~6)를 가짐")
print(df.groupby('GENDER_CD')['AGE_CD'].unique())


---
## 지역키 컬럼 추가 (안전장치)

전국에 이름이 중복되는 시군구가 있다(중구·동구·서구·남구·북구·고성군·강서구 7개 — 예: 강원 고성군 vs 경남 고성군). `CCG_NM`만으로 groupby하면 서로 다른 지역이 하나로 섞일 위험이 있다. SIDO_NM+CCG_NM을 합친 지역키 컬럼을 미리 만들어둔다 — `foreign`은 이 아래에서 df를 그대로 슬라이싱해 만들어지므로 이 컬럼을 자동으로 이어받는다. (실제 안전장치는 이후 모든 지역 단위 집계에서 SIDO_NM과 CCG_NM을 항상 함께 쓰는 것 자체이고, 이 컬럼은 그 작업을 더 쉽게 만들어주는 보조 수단이다.)

In [ ]:
df['REGION_KEY'] = df['SIDO_NM'] + '_' + df['CCG_NM']

dup_check = df[['SIDO_NM', 'CCG_NM']].drop_duplicates()
dup_names = dup_check['CCG_NM'].value_counts()
dup_names = dup_names[dup_names > 1]
print("⚠️ 전국에 이름이 중복되는 시군구:", list(dup_names.index))
print("→ 이후 지역 단위 집계/groupby는 반드시 REGION_KEY(또는 SIDO_NM+CCG_NM 튜플) 기준으로 할 것")


## 3. 업종명 공백 정규화

TP_BUZ_NM 11개 중 3개(편의점·슈퍼마켓·제과점)에 글자 사이 공백이 들어가 있는 걸 확인했다. groupby 자체는 문제없이 되지만(값이 일관되게 그 형태로 들어있어서), 나중에 리포트에 표시하거나 구글트렌드 키워드로 쓸 때 문제가 되므로 정규화된 이름 컬럼을 따로 만들어 둔다. 원본 코드(TP_BUZ_NO)는 그대로 canonical key로 사용.

In [ ]:
df['TP_BUZ_NM_CLEAN'] = df['TP_BUZ_NM'].str.replace(' ', '', regex=False)

check = df[['TP_BUZ_NO', 'TP_BUZ_NM', 'TP_BUZ_NM_CLEAN']].drop_duplicates().sort_values('TP_BUZ_NO')
print(check.to_string(index=False))


## 4. 외국인 세그먼트 추출 (GENDER_CD = '3')

In [ ]:
foreign = df[df['GENDER_CD'] == '3'].copy()

print(f"외국인 세그먼트: {len(foreign):,}행 (전체의 {len(foreign)/len(df)*100:.2f}%)")
print(f"외국인 거래건수 합계(CNT): {foreign['CNT'].sum():,}건  ← 표본 자체는 충분히 큼")
print(f"외국인 소비금액 합계(AMT): {foreign['AMT'].sum():,}원")
print()
print("셀(시군구×업종×월×연령) 단위 CNT 분포 — 이 편차 때문에 최소건수 필터가 필요함:")
print(foreign['CNT'].describe())


## 5. 피벗 테이블 구성

(시군구 × 업종) 를 행으로, 월을 열로 하는 피벗을 AMT/CNT 각각 만든다. 이후 EDA·신호탐지·가설검증 단계에서 전부 이 피벗을 기준으로 진행한다.

In [ ]:
months = sorted(foreign['STRD_YYMM'].unique())

pivot_amt = foreign.pivot_table(
    index=['SIDO_NM', 'CCG_NM', 'TP_BUZ_NM_CLEAN'],
    columns='STRD_YYMM', values='AMT', aggfunc='sum', fill_value=0
)[months]

pivot_cnt = foreign.pivot_table(
    index=['SIDO_NM', 'CCG_NM', 'TP_BUZ_NM_CLEAN'],
    columns='STRD_YYMM', values='CNT', aggfunc='sum', fill_value=0
)[months]

print("피벗 shape (지역×업종 조합 개수, 월 개수):", pivot_amt.shape)
pivot_cnt.head()


## 6. 전국 레벨 baseline 집계

3단계(신호탐지)에서 쓸 "같은 업종의 전국 평균 성장률" 계산을 위해 지금 미리 만들어 둔다. 이걸 나중에 지역별 성장률에서 빼면 계절성이 상쇄된 상대 초과성장률이 나온다(단순 증가율만 쓰면 전북 무주군 같은 계절성 착시가 1위로 잡히는 문제가 있었음).

In [ ]:
national_amt = pivot_amt.groupby(level='TP_BUZ_NM_CLEAN').sum()
national_cnt = pivot_cnt.groupby(level='TP_BUZ_NM_CLEAN').sum()
national_growth = national_amt.pct_change(axis=1)

print("업종별 전국 월간 성장률:")
national_growth.round(3)


## 7. 최소 거래건수 필터

셀 단위 CNT가 최소 11건 ~ 최대 99,532건까지 편차가 커서(중앙값 274건), 필터 없이 랭킹을 매기면 우연히 튄 저건수 셀이 상위권에 낄 위험이 있다. 두 기준을 같이 건다:
- **6개월 누적 건수**: 지역×업종 조합 전체의 규모가 일정 이상이어야 함
- **최근월 건수**: 트렌드 막판(가장 최근 달)에도 최소한의 거래량이 있어야 "지금도 일어나고 있는 신호"로 볼 수 있음 (월별로 전부 다 threshold를 넘기라고 하면 초반에 낮게 시작해서 커지는 진짜 트렌드까지 걸러질 수 있어서, 최근월만 체크)

기준값(200건 / 30건)은 이전 탐색 때 본 분포(25% 지점 78건, 중앙값 274건)를 참고해 잡은 초안이니, EDA 하면서 후보 개수를 보고 조정하면 됨.

In [ ]:
MIN_CUM_CNT = 200      # 6개월 누적 최소 거래건수
MIN_LATEST_CNT = 30    # 최근월(6월) 최소 거래건수

cum_cnt = pivot_cnt.sum(axis=1)
latest_cnt = pivot_cnt[months[-1]]

reliable_mask = (cum_cnt >= MIN_CUM_CNT) & (latest_cnt >= MIN_LATEST_CNT)

print(f"전체 지역×업종 조합: {len(pivot_cnt):,}개")
print(f"필터 통과: {reliable_mask.sum():,}개 ({reliable_mask.mean()*100:.1f}%)")

pivot_amt_reliable = pivot_amt[reliable_mask]
pivot_cnt_reliable = pivot_cnt[reliable_mask]


## 8. 검증: 필터 적용 후에도 기존 플래그십 사례(청송군)가 살아있는지 확인

전처리가 제대로 됐다면, 이전에 찾아둔 경북 청송군(편의점·서양음식·슈퍼마켓 증가 사례)이 필터를 통과해서 남아있어야 한다.

In [ ]:
cs = pivot_cnt_reliable.loc[
    pivot_cnt_reliable.index.get_level_values('CCG_NM') == '청송군'
]
print("청송군 필터 통과 여부:")
cs


## 8-1. 필터 경계선 검증 — "청송군만 확인하는 걸로 충분한가?" (9/15 추가)

바로 위 검증은 청송군이 필터를 통과해서 살아있는지만 본다 — 근데 이건 "필터가 너무 빡빡해서 좋은 신호를 놓치고 있지는 않은가"라는 질문에는 답을 못 해준다(청송군의 헤드라인 3개 업종은 애초에 기준을 여유 있게 넘는 대형 표본이라서). 그래서 제외된 88개 조합 전체를 누적건수 기준 내림차순으로 나열해서, 200이라는 경계선 바로 근처에서 아깝게 탈락한 케이스 중에 진짜 신호처럼 보이는 게 있는지 육안으로 확인한다.

In [ ]:
excluded_combos = pivot_cnt[~reliable_mask].copy()
excluded_cum = cum_cnt[~reliable_mask]
excluded_latest = latest_cnt[~reliable_mask]

boundary_check = pd.DataFrame({
    '1월': excluded_combos[months[0]],
    '6월': excluded_combos[months[-1]],
    '누적건수': excluded_cum,
    '최근월건수': excluded_latest,
})
boundary_check['배율(6월/1월, +1보정)'] = (boundary_check['6월'] + 1) / (boundary_check['1월'] + 1)
boundary_check = boundary_check.sort_values('누적건수', ascending=False)

print(f"필터 제외 총 {len(boundary_check)}개 — 누적건수 상위 20개(200 기준선에 가장 가까운 순):")
print(boundary_check.head(20))

print()
print("청송군의 다른 업종도 필터에 걸렸는지 확인:")
print(boundary_check[boundary_check.index.get_level_values('CCG_NM') == '청송군'])


**확인 결과 (9/15)**: 청송군에도 필터에 걸려 빠진 업종이 있었다 — **제과점**(82누적/42최근월, 15→42건, 2.7배)과 **중국음식**(122누적/27최근월, 11→27건, 2.3배). 즉 바로 위 셀 20의 검증은 청송군 전체가 아니라 헤드라인 3개 업종(편의점·서양음식·슈퍼마켓)만 확인한 것이었다.

경계선(200) 바로 아래에서 아깝게 탈락한 케이스도 있다: **과천시 일식회집**(199건 — 1건 차이로 탈락, 18→60건, 3.2배)·**화성시 병점구 일식회집**(195건, 11→59건, 5배)·**고성군(강원) 일식회집**(111건, 11→39건, 3.3배).

반면 제외된 88개 중 다수는 1월 값이 0이었다가 6월에 갑자기 생긴 전형적 저베이스 노이즈다(예: 장수군 제과점 0→45건 — 배율이 46배로 튀지만 절대 건수 자체가 너무 작아 의미 없음). 이런 건 필터가 의도한 대로 정확히 걸러낸 것.

→ **결론**: 200/30이라는 기준 자체를 바꿀 만한 근거는 부족하다(경계선 근처 후보 수가 적고 대부분 노이즈성). 다만 200 근처 소수 후보(과천시·화성시 병점구·고성군 일식회집 등)와 청송군의 제과점·중국음식은 "표본 기준선 근처라 이번 분석에서는 판단을 보류했다"는 점을 6단계 한계점에 짧게 언급할 가치가 있음 — 필터가 완벽하지 않다는 걸 숨기지 않고 명시하는 것 자체가 이 프로젝트의 방법론적 정직성을 보여주는 부분이기도 하다.

## 9. 결과 저장

다음 단계(EDA, 재랭킹, 가설검증)에서 다시 로드해 쓸 수 있도록 저장한다. 코랩 세션은 끊기면 파일이 날아가니, 다운로드까지 받아두는 걸 추천.

In [ ]:
import pickle

preprocessed = {
    'months': months,
    'pivot_amt': pivot_amt,
    'pivot_cnt': pivot_cnt,
    'national_amt': national_amt,
    'national_cnt': national_cnt,
    'reliable_mask': reliable_mask,
    'pivot_amt_reliable': pivot_amt_reliable,
    'pivot_cnt_reliable': pivot_cnt_reliable,
}

# 구글 드라이브의 BC카드 폴더에 저장 — 다음 세션에서 재업로드 없이 바로 불러올 수 있음
drive_out_path = '/content/drive/MyDrive/BC카드/bc_preprocessed.pkl'
with open(drive_out_path, 'wb') as f:
    pickle.dump(preprocessed, f)
print("저장 완료:", drive_out_path)

# 내 컴퓨터로도 따로 다운받고 싶으면 아래 두 줄 주석 해제
# from google.colab import files
# files.download(drive_out_path)


## 부록. 3단계 미리보기 — 필터 적용 후 재랭킹

전처리가 3단계(신호탐지)에 실제로 영향을 주는지 바로 확인해본다. 이 셀은 본격적인 3단계 작업(다음 세션)에서 다시 다듬을 예정이고, 지금은 필터가 잘 작동했는지 sanity check 용도.

In [ ]:
def relative_growth_score(pivot_amt_sub, national_amt_full, months, n_recent=2):
    growth = pivot_amt_sub.pct_change(axis=1)
    nat_growth = national_amt_full.pct_change(axis=1)

    rel = growth.copy()
    for buz in national_amt_full.index:
        idx = rel.index.get_level_values('TP_BUZ_NM_CLEAN') == buz
        rel.loc[idx] = growth.loc[idx].values - nat_growth.loc[buz].values

    return rel[months[-n_recent:]].mean(axis=1).sort_values(ascending=False)

top_candidates = relative_growth_score(pivot_amt_reliable, national_amt, months)
print("필터 적용 후 상대 초과성장률 상위 15개 (지역×업종):")
top_candidates.head(15)


---
# 3단계. 신호 탐지 — CNT 기준 최종 스코어링 (9/14)

AMT(금액) 기준과 CNT(건수) 기준을 각각 돌려보니 순위가 달라졌다(객단가는 높은데 방문객 수는 적을 수 있어서 당연함). "아직 안 알려진 곳을 방문객 수로 먼저 포착한다"는 서사에 더 맞는 **CNT 기준을 최종 채택**한다. 함수는 위 부록에서 정의한 `relative_growth_score`를 그대로 재사용(범용 함수라 AMT든 CNT든 넣을 수 있음).

In [ ]:
cnt_scores = relative_growth_score(pivot_cnt_reliable, national_cnt, months)

print("CNT 기준 상대 초과성장 상위 20개:")
cnt_scores.head(20)


## 6개월 추이 육안 검수로 확인된 아티팩트 제외

상위 후보를 6개월 추이로 하나씩 눈으로 확인한 결과, 자동 스코어만으로는 못 거르는 두 가지 함정이 있었다:
- **저베이스 반등**: 무주군 스넥(476→540→59→43→38→76) — 겨울 스키시즌 급감 후 낮은 바닥에서 살짝 반등한 걸 "최근 2개월 증가율"이 과대평가
- **지역축제**: 곡성군 서양음식(5월 곡성세계장미축제), 포천시·음성군도 마찬가지로 확인 — 단, 포천시·음성군은 축제 달을 빼고 재계산해도 순수 신호가 남아있어서 후보로 복귀시킴(아래 별도 셀)

재현 가능하도록 제외/조정 목록을 명시적으로 남겨둔다.

In [ ]:
# 육안 검수로 확정한 최종 제외 목록 (지역, 업종): 사유
EXCLUDED_ARTIFACTS = {
    ('전북특별자치도', '무주군', '스넥'): '저베이스 반등 — 겨울 스키시즌 급감 후 낮은 바닥에서 반등',
    ('전라남도', '곡성군', '서양음식'): '지역축제(곡성세계장미축제 2026.5.22~31) 효과가 대부분',
}

cnt_scores_clean = cnt_scores.drop(index=[k for k in EXCLUDED_ARTIFACTS if k in cnt_scores.index])
print(f"제외 {len(EXCLUDED_ARTIFACTS)}건 처리 후 상위 15개:")
cnt_scores_clean.head(15)


---
# 지역 단위 재집계 — "청송군만 특별한가?" 재검증 (9/14)

상위 20개(개별 지역×업종 조합)만 보는 방식은 편향될 수 있다는 문제제기에 따라, **지역(시군구) 단위로 묶어서 몇 개 업종에서 동시에 양수 신호(score>0.1)가 뜨는지** 전체 2,111개 조합에 대해 재집계한다. 반드시 `SIDO_NM`+`CCG_NM`을 함께 묶어야 위에서 확인한 동명 시군구 문제를 피할 수 있다.

In [ ]:
sdf = cnt_scores.reset_index()
sdf.columns = ['SIDO_NM', 'CCG_NM', 'TP_BUZ_NM_CLEAN', 'score']
sdf['positive'] = sdf['score'] > 0.1

region_summary = sdf.groupby(['SIDO_NM', 'CCG_NM']).agg(
    양수업종수=('positive', 'sum'),
    총업종수=('TP_BUZ_NM_CLEAN', 'count'),
    평균점수=('score', 'mean'),
    최고점수=('score', 'max'),
).sort_values(['양수업종수', '평균점수'], ascending=False)

print("여러 업종에서 동시에 양수 신호를 보이는 지역 Top 15:")
print(region_summary.head(15))
print()
print("참고: 청송군 상세")
print(sdf[sdf['CCG_NM'] == '청송군'].sort_values('score', ascending=False))


## 교차검증 — 다른 렌즈로도 같은 지역이 나오는가

"이 지역들만 봐도 되는가"에 답하기 위해, 서로 다른 두 가지 기준으로 더 교차검증한다:
- **절대 순증가량**: 1월 대비 6월 CNT가 얼마나 늘었는가(% 아니고 절대 건수) — 원래 큰 도시가 나오면, 우리가 % 성장률 기반으로 접근한 게 맞았다는 역검증이 됨
- **양수비율**: 업종 개수와 무관하게, 가진 업종 중 몇 %가 양수인가 — 총 업종 수가 적은 지역도 놓치지 않기 위함

In [ ]:
region_cnt_sum = pivot_cnt_reliable.groupby(level=['SIDO_NM', 'CCG_NM']).sum()
abs_growth = region_cnt_sum[months[-1]] - region_cnt_sum[months[0]]
print("=== 절대 순증가량 Top 10 (규모가 큰 도시 위주여야 정상) ===")
print(abs_growth.sort_values(ascending=False).head(10))
print()

region_ratio = region_summary.copy()
region_ratio['비율'] = region_ratio['양수업종수'] / region_ratio['총업종수']
region_ratio = region_ratio[region_ratio['총업종수'] >= 2].sort_values(['비율', '평균점수'], ascending=False)
print("=== 양수비율 Top 15 ===")
print(region_ratio.head(15))


---
# 축제 효과 분리 — 제외 대신 "순수 신호가 남는가" 확인 (9/14)

계절성을 다뤘던 방식(전국평균을 빼서 순수 신호만 남기기)과 같은 논리로, 축제로 확인된 케이스도 통째로 버리지 않고 축제 달만 빼고 재계산한다. 실제로 포천시는 축제를 빼도 신호가 절반 이상 남아서 후보로 복귀시켰다.

In [ ]:
def excess_growth_excluding(row, nat_row, from_idx, to_idx):
    '''축제 등으로 오염된 달을 건너뛰고 from_idx -> to_idx 구간만으로 상대초과성장(%) 계산'''
    g_region = row.iloc[to_idx] / row.iloc[from_idx] - 1
    g_nat = nat_row.iloc[to_idx] / nat_row.iloc[from_idx] - 1
    return g_region - g_nat

festival_cases = [
    ('전라남도', '곡성군', '서양음식', 3, 5, '곡성세계장미축제(2026.5.22~31)'),
    ('경기도', '포천시', '일식회집', 3, 5, '한탄강 봄 가든페스타(2026.5.1~6.7)'),
    ('충청북도', '음성군', '대형할인점', 3, 4, '음성품바축제(2026.6.10~14, 마지막 달이라 축제 이전 구간만 확인)'),
]

for sido, ccg, buz, i0, i1, festival in festival_cases:
    row = pivot_cnt_reliable.loc[(sido, ccg, buz)]
    nat = national_cnt.loc[buz]
    excess = excess_growth_excluding(row, nat, i0, i1)
    original = cnt_scores.loc[(sido, ccg, buz)]
    print(f"{ccg} {buz} — {festival}")
    print(f"  원래 점수: {original:.3f} / 축제 달 제외 후 상대초과성장: {excess*100:.1f}%p")
    print(f"  → {'후보 복귀 (순수 신호 남음)' if excess > 0.1 else '제외 유지 (축제 효과가 대부분)'}")
    print()


---
# 3단계 결과 저장

1단계 pkl과 별도로, 오늘 확정한 3단계 신호탐지 결과를 저장한다.

In [ ]:
import pickle

stage3_results = {
    'cnt_scores': cnt_scores,
    'cnt_scores_clean': cnt_scores_clean,
    'excluded_artifacts': EXCLUDED_ARTIFACTS,
    'region_summary': region_summary,
}

drive_out_path2 = '/content/drive/MyDrive/BC카드/bc_stage3_signals.pkl'
with open(drive_out_path2, 'wb') as f:
    pickle.dump(stage3_results, f)
print("3단계 결과 저장 완료:", drive_out_path2)
